# scVI Integration 
Trains scVI on the merged, QC-filtered counts, saves the latent embeddings, then uses one of those embeddings
to recompute neighbors/UMAP and re-cluster with Leiden.

## Setup (Colab only)

Skip this cell if running locally / on a cluster with dependencies already installed.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/

# !pip install --quiet scanpy scikit-misc scvi-tools==1.2.1 igraph leidenalg

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch

## Config

In [ ]:
input_path = "S2_counts_preprocessed_adata_merge.h5ad"
output_dir = "./scvi_results"
output_path = "S3_scvi_adata_merge.h5ad"

n_latent_values = [10, 20, 30]
batch_key = "ID"
categorical_covariate_keys = ["batchseq"]
continuous_covariate_keys = ["percent_mito"]

# which trained latent representation to use for downstream clustering/UMAP
cluster_latent_key = "X_scVI_n_latent_10"
n_neighbors = 40
leiden_resolution = 0.2

os.makedirs(output_dir, exist_ok=True)
scvi.settings.seed = 0
print("scvi-tools version:", scvi.__version__)

## Load and preprocess data

In [ ]:
adata = sc.read(input_path)  # non-normalized raw counts
adata.layers["raw_counts"] = adata.X.copy()  # preserve raw counts before normalizing

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata

## Train scVI on global

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="raw_counts",
    batch_key=batch_key,
    categorical_covariate_keys=categorical_covariate_keys,
    continuous_covariate_keys=continuous_covariate_keys,
)

models = {}
for n_latent in n_latent_values:
    print("Training scVI model with n_latent:", n_latent)
    models[n_latent] = scvi.model.SCVI(adata, n_latent=n_latent)
    models[n_latent].train()

## Extract and save latent embeddings

In [ ]:
for n_latent in n_latent_values:
    latent_key = f"X_scVI_n_latent_{n_latent}"
    adata.obsm[latent_key] = models[n_latent].get_latent_representation()

    latent_df = pd.DataFrame(adata.obsm[latent_key], index=adata.obs.index)
    out_path = os.path.join(output_dir, f"allgenes_obsm_with_scVI_latent_representation_n_{n_latent}.csv")
    latent_df.to_csv(out_path)
    print(f"Saved latent representation (n_latent={n_latent}) to {out_path}")

## Save integrated output

In [ ]:
adata.X = adata.layers["raw_counts"].copy()
if "log1p" in adata.uns:
    del adata.uns["log1p"]

adata.write(output_path)
adata

## Downstream clustering using scVI latent representation

Recomputes neighbors/UMAP directly from the chosen scVI embedding (`cluster_latent_key`) rather than from PCA, then re-clusters with Leiden.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["logcounts"] = adata.X.copy()

In [ ]:
sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=cluster_latent_key)
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added="leiden_scvi", resolution=leiden_resolution)

sc.pl.umap(
    adata, color=["leiden_scvi", "phase", "n_genes", "percent_mito", "ID", "group2"],
    ncols=3, wspace=0.3, size=15, color_map="OrRd",
)